In [5]:
import os
import pandas as pd
from tqdm import tqdm
import json
import requests
from PIL import Image
from io import BytesIO
#'../dataset'
# Configuration
DATASET_FOLDER = '/Users/bhavinbaldota/Library/CloudStorage/OneDrive-MSFT/Desktop/Amazon L1/dataset'
IMAGE_FOLDER = '/Users/bhavinbaldota/Library/CloudStorage/OneDrive-MSFT/Desktop/Amazon L1/images/train'
BATCH_SIZE = 100
CHECKPOINT_FILE = 'download_checkpoint.json'

# Create image folder
os.makedirs(IMAGE_FOLDER, exist_ok=True)

# Load data
train = pd.read_csv(os.path.join(DATASET_FOLDER, 'train.csv'))
total_images = len(train)
total_batches = (total_images + BATCH_SIZE - 1) // BATCH_SIZE

print(f"Total images: {total_images}")
print(f"Total batches: {total_batches}")

# Custom download function with error handling
def download_single_image(url, save_path, timeout=10):
    try:
        response = requests.get(url, timeout=timeout)
        response.raise_for_status()
        img = Image.open(BytesIO(response.content))
        img.save(save_path)
        return True
    except Exception as e:
        return False

def download_batch(image_links, folder, start_idx):
    success_count = 0
    failed_count = 0
    
    for i, link in enumerate(tqdm(image_links, desc="Downloading")):
        img_idx = start_idx + i
        save_path = os.path.join(folder, f'image_{img_idx}.jpg')
        
        # Skip if already downloaded
        if os.path.exists(save_path):
            success_count += 1
            continue
            
        if download_single_image(link, save_path):
            success_count += 1
        else:
            failed_count += 1
    
    return success_count, failed_count

# Load/save checkpoint
def load_checkpoint():
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE, 'r') as f:
            return json.load(f)
    return {'last_completed_batch': -1}

def save_checkpoint(batch_num):
    with open(CHECKPOINT_FILE, 'w') as f:
        json.dump({'last_completed_batch': batch_num}, f)

# Get starting batch
checkpoint = load_checkpoint()
start_batch = checkpoint['last_completed_batch'] + 1

print(f"\nLast completed batch: {checkpoint['last_completed_batch']}")
user_input = input(f"Press Enter to continue from batch {start_batch}, or enter batch number: ").strip()
if user_input:
    start_batch = int(user_input)

print(f"\nStarting from batch {start_batch}/{total_batches-1}\n")

# Download in batches
for batch_num in range(start_batch, total_batches):
    start_idx = batch_num * BATCH_SIZE
    end_idx = min(start_idx + BATCH_SIZE, total_images)
    
    print(f"{'='*60}")
    print(f"Batch {batch_num}/{total_batches-1} | Images {start_idx}-{end_idx-1}")
    print(f"{'='*60}")
    
    batch_links = train['image_link'].iloc[start_idx:end_idx].tolist()
    
    success, failed = download_batch(batch_links, IMAGE_FOLDER, start_idx)
    
    print(f"✓ Success: {success} | ✗ Failed: {failed}")
    save_checkpoint(batch_num)
    print()
    from IPython.display import clear_output
    if (batch_num+1) % 10 == 0:
        clear_output(wait=True)

print("="*60)
print("✓ All batches completed!")





Batch 450/749 | Images 45000-45099


Downloading: 100%|██████████| 100/100 [00:53<00:00,  1.87it/s]


✓ Success: 100 | ✗ Failed: 0

Batch 451/749 | Images 45100-45199


Downloading: 100%|██████████| 100/100 [00:48<00:00,  2.06it/s]


✓ Success: 100 | ✗ Failed: 0

Batch 452/749 | Images 45200-45299


Downloading: 100%|██████████| 100/100 [00:50<00:00,  1.96it/s]


✓ Success: 100 | ✗ Failed: 0

Batch 453/749 | Images 45300-45399


Downloading:  40%|████      | 40/100 [00:21<00:31,  1.90it/s]


KeyboardInterrupt: 